# SIT307 – 8.1D Machine Learning Mini Project
## Sydney Housing Price Prediction
**Sara Zawad | Deakin University**

In this project I'm trying to predict property sale prices in three Sydney suburbs —
Parramatta, Chatswood, and Bondi. I'll look at the data, train some models,
and see which one works best.

## Part 1 – Setup and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data/sydney_housing.csv')
print(f"Total properties: {len(df)}")
print(df['suburb'].value_counts())

In [ ]:
# Quick look at the data
df.head()

In [ ]:
# Price summary by suburb
df.groupby('suburb')['sale_price'].describe().round(0)

## Part 2 – Exploring the Data

Before training any models I want to understand what the data looks like and
which features seem to affect the price most.

In [ ]:
COLORS = {'Parramatta': '#4C72B0', 'Chatswood': '#55A868', 'Bondi': '#DD8452'}

# Price distribution per suburb
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (sub, grp) in zip(axes, df.groupby('suburb')):
    ax.hist(grp['sale_price']/1e6, bins=12, color=COLORS[sub], edgecolor='white')
    ax.set_title(sub, fontweight='bold')
    ax.set_xlabel('Price ($M)')
    ax.set_ylabel('Count')
    ax.axvline(grp['sale_price'].median()/1e6, color='red', linestyle='--',
               label=f"Median ${grp['sale_price'].median()/1e6:.2f}M")
    ax.legend(fontsize=8)
plt.suptitle('Sale Price by Suburb', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Bondi is the most expensive and has the widest price range.
# Parramatta is the most affordable and the tightest.

In [ ]:
# Price by property type
pivot = df.groupby(['suburb','property_type'])['sale_price'].median().unstack()/1e6
pivot.plot(kind='bar', figsize=(9,4), edgecolor='white', color=['#4C72B0','#DD8452','#55A868'])
plt.ylabel('Median Price ($M)')
plt.title('Median Price by Suburb & Property Type', fontweight='bold')
plt.xticks(rotation=0)
plt.legend(title='Type')
plt.tight_layout()
plt.show()

# Houses cost a lot more than apartments — especially in Bondi.

In [ ]:
# Correlation heatmap — which features relate most to price?
num_cols = ['bedrooms','bathrooms','parking','land_size_sqm','floor_area_sqm',
            'property_age_years','distance_to_cbd_km','school_rating','sale_price']
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, mask=np.triu(np.ones((9,9), dtype=bool)), ax=ax)
ax.set_title('Feature Correlations', fontweight='bold')
plt.tight_layout()
plt.show()

# distance_to_cbd has a strong negative correlation (-0.54) — closer to CBD = more expensive.
# land_size and floor_area both positively correlate with price.

In [ ]:
# Floor area vs price
fig, ax = plt.subplots(figsize=(8,4))
for sub, grp in df.groupby('suburb'):
    ax.scatter(grp['floor_area_sqm'], grp['sale_price']/1e6,
               label=sub, color=COLORS[sub], alpha=0.7, edgecolors='white', s=50)
ax.set_xlabel('Floor Area (sqm)')
ax.set_ylabel('Price ($M)')
ax.set_title('Floor Area vs Sale Price', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### Feature Engineering

I need to prepare the data for the models:
- **Encode** suburb and property_type (text → numbers)
- **Create** a couple of extra features
- **Split** into train and test sets

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

df2 = df.copy()

le_suburb = LabelEncoder()
le_type   = LabelEncoder()
df2['suburb_enc']        = le_suburb.fit_transform(df2['suburb'])
df2['property_type_enc'] = le_type.fit_transform(df2['property_type'])
df2['bed_bath_ratio']    = df2['bedrooms'] / df2['bathrooms'].replace(0, 1)
df2['is_new']            = (df2['property_age_years'] <= 5).astype(int)

FEATURES = ['suburb_enc','property_type_enc','bedrooms','bathrooms','parking',
            'land_size_sqm','floor_area_sqm','property_age_years',
            'distance_to_cbd_km','school_rating','bed_bath_ratio','is_new']

X = df2[FEATURES]
y = df2['sale_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training: {len(X_train)} properties")
print(f"Testing:  {len(X_test)} properties")

In [ ]:
# Which features matter most? Quick check with Random Forest
from sklearn.ensemble import RandomForestRegressor

rf_check = RandomForestRegressor(n_estimators=100, random_state=42)
rf_check.fit(X_train, y_train)

importances = pd.Series(rf_check.feature_importances_, index=FEATURES).sort_values(ascending=False)
importances.plot(kind='barh', figsize=(8,4), color='#4C72B0', edgecolor='white')
plt.gca().invert_yaxis()
plt.xlabel('Importance')
plt.title('Which Features Matter Most?', fontweight='bold')
plt.tight_layout()
plt.show()

print("Top 3:", importances.head(3).index.tolist())

## Part 3 – Training 3 Models

I'm going to train:
1. **Linear Regression** — simple baseline
2. **Decision Tree** — splits data into rules
3. **Random Forest** — many trees averaged together

I'll use **5-fold cross-validation** to get reliable scores, not just a single train/test split.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_score

def run_model(model, X_tr, y_tr, X_te, y_te, name):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    mae = mean_absolute_error(y_te, pred)
    r2  = r2_score(y_te, pred)
    kf  = KFold(n_splits=5, shuffle=True, random_state=42)
    cv  = cross_val_score(model, X, y, cv=kf, scoring='r2')
    print(f"--- {name} ---")
    print(f"  Test R²:      {r2:.3f}")
    print(f"  Test MAE:     ${mae:,.0f}")
    print(f"  CV R² (5-fold): {cv.mean():.3f} ± {cv.std():.3f}")
    return pred

# Linear Regression needs scaled features
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

lr_pred = run_model(LinearRegression(), X_tr_sc, y_train, X_te_sc, y_test, 'Linear Regression')

In [ ]:
dt_pred = run_model(
    DecisionTreeRegressor(max_depth=6, random_state=42),
    X_train, y_train, X_test, y_test, 'Decision Tree'
)

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_pred  = run_model(rf_model, X_train, y_train, X_test, y_test, 'Random Forest')

### Bias-Variance Trade-off (Decision Tree depth experiment)

One important concept in ML is **bias vs variance**:
- **High bias** = model is too simple, underfits the data
- **High variance** = model memorises training data, performs badly on new data

I can see this clearly by changing the Decision Tree depth:

In [ ]:
from sklearn.tree import DecisionTreeRegressor

train_r2, test_r2 = [], []
for d in range(1, 15):
    m = DecisionTreeRegressor(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_r2.append(r2_score(y_train, m.predict(X_train)))
    test_r2.append(r2_score(y_test,  m.predict(X_test)))

plt.figure(figsize=(8,4))
plt.plot(range(1,15), train_r2, 'o-', label='Train R²', color='#4C72B0')
plt.plot(range(1,15), test_r2,  's-', label='Test R²',  color='#DD8452')
plt.fill_between(range(1,15), train_r2, test_r2, alpha=0.15, color='grey', label='Overfitting gap')
plt.xlabel('Tree Depth')
plt.ylabel('R²')
plt.title('Decision Tree: Bias vs Variance', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# At depth 1-3: underfitting (high bias). At depth 10+: overfitting (high variance).
# Depth 6 is a good middle ground.

In [ ]:
# Model comparison chart
models = ['Linear Regression', 'Decision Tree', 'Random Forest']
r2s  = [r2_score(y_test, lr_pred), r2_score(y_test, dt_pred), r2_score(y_test, rf_pred)]
maes = [mean_absolute_error(y_test, lr_pred), mean_absolute_error(y_test, dt_pred),
        mean_absolute_error(y_test, rf_pred)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
colors = ['#4C72B0','#55A868','#DD8452']

bars = ax1.bar(models, r2s, color=colors, edgecolor='white', width=0.5)
ax1.set_title('R² Score (higher = better)', fontweight='bold')
ax1.set_ylim(0, 1.05)
for b, v in zip(bars, r2s):
    ax1.text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.3f}', ha='center', fontweight='bold')

bars2 = ax2.bar(models, [v/1000 for v in maes], color=colors, edgecolor='white', width=0.5)
ax2.set_title('MAE in $000s (lower = better)', fontweight='bold')
ax2.set_ylabel('$000s')
for b, v in zip(bars2, maes):
    ax2.text(b.get_x()+b.get_width()/2, v/1000+1, f'${v/1000:.0f}k', ha='center', fontweight='bold')

plt.suptitle('Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Winner: Random Forest — lowest MAE and highest CV R²")

In [ ]:
# Actual vs Predicted (Random Forest)
plt.figure(figsize=(6,5))
plt.scatter(y_test/1e6, rf_pred/1e6, alpha=0.7, color='#4C72B0', edgecolors='white', s=60)
mn = min(y_test.min(), rf_pred.min())/1e6
mx = max(y_test.max(), rf_pred.max())/1e6
plt.plot([mn, mx], [mn, mx], 'r--', label='Perfect prediction')
plt.xlabel('Actual Price ($M)')
plt.ylabel('Predicted Price ($M)')
plt.title('Random Forest: Actual vs Predicted', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

## Part 4 – 5 Worst Prediction Errors

I want to look at where the model got it most wrong and understand why.

In [ ]:
errors = X_test.copy()
errors['actual']    = y_test.values
errors['predicted'] = rf_pred
errors['abs_error'] = abs(errors['predicted'] - errors['actual'])
errors['pct_error'] = (errors['abs_error'] / errors['actual'] * 100).round(1)
errors['suburb']    = le_suburb.inverse_transform(errors['suburb_enc'].astype(int))
errors['type']      = le_type.inverse_transform(errors['property_type_enc'].astype(int))
worst = errors.sort_values('abs_error', ascending=False).head(5).reset_index(drop=True)

for i, r in worst.iterrows():
    print(f"#{i+1} {r['suburb']} {r['type']} {int(r['bedrooms'])}bed")
    print(f"   Actual: ${r['actual']/1e6:.3f}M  |  Predicted: ${r['predicted']/1e6:.3f}M  |  Error: {r['pct_error']:.1f}%")

In [ ]:
# Bar chart of worst 5
fig, ax = plt.subplots(figsize=(10,4))
w = 0.35
idx = range(5)
ax.bar([i-w/2 for i in idx], worst['actual']/1e6, w, label='Actual', color='#4C72B0', edgecolor='white')
ax.bar([i+w/2 for i in idx], worst['predicted']/1e6, w, label='Predicted', color='#DD8452', edgecolor='white')
labels = [(r['suburb']+' '+r['type']+' '+str(int(r['bedrooms']))+'bed') for _,r in worst.iterrows()]
ax.set_xticks(list(idx)); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('Price ($M)')
ax.set_title('5 Worst Prediction Errors', fontweight='bold')
ax.legend()
for i, r in worst.iterrows():
    ax.text(i, max(r['actual'],r['predicted'])/1e6+0.05, f"{r['pct_error']:.1f}%",
            ha='center', fontsize=9, color='red', fontweight='bold')
plt.tight_layout()
plt.show()

### Why did the model get these wrong?

**#1 – Bondi House 5bed (49.2% error):** The biggest mistake. Model predicted $2.85M but actual was $5.6M.
There are only 8 Bondi houses in the dataset — not enough for the model to learn the top end of Bondi prices.

**#2 & #3 – Chatswood Houses (10–16% error):** Chatswood houses vary a lot depending on exact street and renovation.
The model only knows the suburb average, not fine details within the suburb.

**#4 – Parramatta Townhouse (18% error):** Model overestimated because it treats all Parramatta properties as
equally located at ~23km from CBD, but some are closer to the station than others.

**#5 – Chatswood Apartment 4bed (15% error):** 4-bed apartments in Chatswood are rare, so the model doesn't
have enough examples to price them accurately.

**Main lesson:** The model struggles most with rare/extreme properties and doesn't know the difference
*within* a suburb (street-level location).

## Part 5 – Web App & Reflection

### Web App

I built a Streamlit app for price predictions. Run it with:
```
streamlit run app/app.py
```
Enter the property details and it gives you a predicted price plus a confidence range.

### Save model for the app

In [ ]:
import pickle, os
os.makedirs('app', exist_ok=True)

bundle = {
    'model': rf_model,
    'features': FEATURES,
    'le_suburb': le_suburb,
    'le_type': le_type,
    'mae': mean_absolute_error(y_test, rf_pred),
}
with open('app/model.pkl', 'wb') as f:
    pickle.dump(bundle, f)
print("Saved! MAE =", f"${mean_absolute_error(y_test, rf_pred):,.0f}")

### Reflection

This was a good project to practice the full ML workflow from start to finish.

A few things I noticed:

- **Random Forest was clearly the best model.** It had the lowest average error ($229k) and the highest cross-validation R² (0.854). Linear regression was too simple for this kind of data. Decision Trees were good but unstable.

- **Location matters more than property size.** The suburb and distance to CBD were the top features. A big apartment in Parramatta still costs less than a small apartment in Bondi.

- **The bias-variance tradeoff experiment was really useful.** Seeing the training R² go to 1.0 at deep trees while the test R² crashed made overfitting very clear.

- **If I had more time**, I'd add more data (especially Bondi houses), include street-level features like distance to beach or station, and try gradient boosting (XGBoost) which often does even better than Random Forest.